In [1]:
import pandas as pd
from pymongo import MongoClient

# connect to database
client = MongoClient('mongo', 27017)
db = client.f1_2025  
print(db)

Database(MongoClient(host=['mongo:27017'], document_class=dict, tz_aware=False, connect=True), 'f1_2025')


In [2]:
collections = db.list_collection_names()
print("Collection presenti:", collections)

Collection presenti: ['races', 'circuits', 'drivers', 'constructors', 'results']


In [3]:
def import_csv_to_mongo(csv_file, collection_name):
    df = pd.read_csv(csv_file)
    df = df.drop_duplicates()  # delete duplicates
    data_dict = df.to_dict("records")

    # empty collection
    collection = db[collection_name]
    collection.delete_many({})
    
    collection.insert_many(data_dict)
    print(f"{len(data_dict)} documenti inseriti in {collection_name}")

In [4]:
csv_files = {
    "drivers": "../data_integration/drivers.csv",
    "constructors": "../data_integration/constructors.csv",
    "circuits": "../data_integration/circuits.csv",
    "races": "../data_cleaning/races.csv",
    "results": "../data_integration/results.csv"
}

for collection_name, file in csv_files.items():
    import_csv_to_mongo(file, collection_name)

21 documenti inseriti in drivers
10 documenti inseriti in constructors
24 documenti inseriti in circuits
24 documenti inseriti in races
599 documenti inseriti in results


## **First query**

In [5]:
driver_name = "Charles"
driver_surname = "Leclerc"

pipeline = [
    # match driver 
    {"$match": {"name": driver_name, "surname": driver_surname}},

    # filter driver
    {"$lookup": {
        "from": "results",
        "let": {"drvId": "$driverId"},
        "pipeline": [
            {"$match": {"$expr": {"$eq": ["$driverId", "$$drvId"]}}}
        ],
        "as": "results"
    }},

    # races
    {"$lookup": {
        "from": "races",
        "localField": "results.raceId",
        "foreignField": "raceId",
        "as": "races"
    }},

    # circuits
    {"$lookup": {
        "from": "circuits",
        "localField": "races.circuitId",
        "foreignField": "circuitId",
        "as": "circuits"
    }},

    # 2025
    {"$addFields": {
        "totalRaces": {"$size": "$results"},
        "winsLatest": {"$size": {"$filter": {"input": "$results", "cond": {"$eq": ["$$this.position", 1]}}}},
        "podiumsLatest": {"$size": {"$filter": {"input": "$results", "cond": {"$lte": ["$$this.position", 3]}}}},
        "pointsLatest": {"$sum": "$results.points"},
        "streetRaces": {"$size": {"$filter": {"input": "$circuits", "cond": {"$eq": ["$$this.type", "Street circuit"]}}}},
        "raceRaces": {"$size": {"$filter": {"input": "$circuits", "cond": {"$eq": ["$$this.type", "Race circuit"]}}}},
        "avgPositionLatest": {"$avg": "$results.position"},
        "seasonsCount": {"$size": {"$split": ["$seasons", "; "]}}
    }},

    # constructor
    {"$lookup": {
        "from": "constructors",
        "localField": "results.constructorId",
        "foreignField": "constructorId",
        "as": "teams"
    }},
    {"$addFields": {
        "team": {"$arrayElemAt": ["$teams.constructor", -1]}
    }},

    # final project
    {"$project": {
        "_id": 0,
        "driverName": "$name",
        "driverSurname": "$surname",
        "dateOfBirth": 1,
        "nationality": 1,
        "team": 1,
        "seasons": "$seasonsCount",
        "totalRaces": 1,
        "streetRaces": 1,
        "raceRaces": 1,
        "winsLatest": 1,
        "podiumsLatest": 1,
        "pointsLatest": 1,
        "avgPositionLatest": 1,
        # Overall career dal driver
        "winsOverall": "$wins",
        "podiumsOverall": "$podiums",
        "polesOverall": "$poles",
        "pointsOverall": "$points",
        "championships": "$championships"
    }}
]

summary = list(db.drivers.aggregate(pipeline))
driver = summary[0] if summary else {}

print(f"DRIVER SUMMARY:")
print(f"Name           : {driver.get('driverName','Unknown')}")
print(f"Surname        : {driver.get('driverSurname','Unknown')}")
print(f"Date of Birth  : {driver.get('dateOfBirth','Unknown')}")
print(f"Nationality    : {driver.get('nationality','Unknown')}")
print(f"Team (Last)    : {driver.get('team','Unknown')}")
print(f"Seasons        : {driver.get('seasons',0)}")

print("\nLatest Season")
print(f"Total races       : {driver.get('totalRaces',0)}")
print(f"Street circuits   : {driver.get('streetRaces',0)}")
print(f"Race circuits     : {driver.get('raceRaces',0)}")
print(f"Points            : {driver.get('pointsLatest',0)}")
print(f"Wins              : {driver.get('winsLatest',0)}")
print(f"Podiums           : {driver.get('podiumsLatest',0)}")
print(f"Average position  : {driver.get('avgPositionLatest',0):.2f}")

print("\nOverall performances")
print(f"Wins          : {driver.get('winsOverall',0)}")
print(f"Podiums       : {driver.get('podiumsOverall',0)}")
print(f"Poles         : {driver.get('polesOverall',0)}")
print(f"Total points  : {driver.get('pointsOverall',0)}")
print(f"Championships : {driver.get('championships',0)}")

DRIVER SUMMARY:
Name           : Charles
Surname        : Leclerc
Date of Birth  : 1997-10-16
Nationality    : Monaco
Team (Last)    : Ferrari
Seasons        : 1

Latest Season
Total races       : 30
Street circuits   : 8
Race circuits     : 16
Points            : 242
Wins              : 0
Podiums           : 7
Average position  : 7.13

Overall performances
Wins          : 8
Podiums       : 50
Poles         : 27
Total points  : 1672.0
Championships : 0


## **Second query**

In [6]:
def delta_summary_by_grid():
    pipeline = [
        # grid and position
        {
            "$project": {
                "grid": 1,
                "delta": {"$subtract": ["$grid", "$position"]},
            }
        },
        # calculate avg
        {
            "$group": {
                "_id": "$grid",
                "avgDelta": {"$avg": "$delta"},
                "avgAbsDelta": {"$avg": {"$abs": "$delta"}},
                "count": {"$sum": 1}
            }
        },
        {
            "$sort": {"_id": 1}
        }
    ]
    return list(db.results.aggregate(pipeline))

summary = delta_summary_by_grid()
print("PERFORMANCE BY GRID POSITION")
for row in summary:
    print(f"Grid {int(row['_id'])}: Avg Gain/Loss={row['avgDelta']:.2f}, "
          f"Avg Abs Change={row['avgAbsDelta']:.2f} (n={row['count']} races)")

PERFORMANCE BY GRID POSITION
Grid 1: Avg Gain/Loss=-1.33, Avg Abs Change=1.33 (n=30 races)
Grid 2: Avg Gain/Loss=-1.43, Avg Abs Change=1.77 (n=30 races)
Grid 3: Avg Gain/Loss=-2.00, Avg Abs Change=3.00 (n=30 races)
Grid 4: Avg Gain/Loss=-1.07, Avg Abs Change=1.93 (n=30 races)
Grid 5: Avg Gain/Loss=-1.30, Avg Abs Change=2.50 (n=30 races)
Grid 6: Avg Gain/Loss=-3.67, Avg Abs Change=4.47 (n=30 races)
Grid 7: Avg Gain/Loss=-1.27, Avg Abs Change=3.13 (n=30 races)
Grid 8: Avg Gain/Loss=-2.00, Avg Abs Change=3.67 (n=30 races)
Grid 9: Avg Gain/Loss=-2.60, Avg Abs Change=4.13 (n=30 races)
Grid 10: Avg Gain/Loss=-0.40, Avg Abs Change=2.60 (n=30 races)
Grid 11: Avg Gain/Loss=-0.93, Avg Abs Change=3.80 (n=30 races)
Grid 12: Avg Gain/Loss=-1.07, Avg Abs Change=2.87 (n=30 races)
Grid 13: Avg Gain/Loss=0.00, Avg Abs Change=3.53 (n=30 races)
Grid 14: Avg Gain/Loss=-0.47, Avg Abs Change=3.33 (n=30 races)
Grid 15: Avg Gain/Loss=1.57, Avg Abs Change=3.23 (n=30 races)
Grid 16: Avg Gain/Loss=1.10, Avg Abs 

## **Third query**

In [7]:
def dnf_street_vs_permanent():
    pipeline = [
        # races
        {
            "$lookup": {
                "from": "races",
                "localField": "round",
                "foreignField": "round",
                "as": "race"
            }
        },
        {"$unwind": "$race"},

        # circuits
        {
            "$lookup": {
                "from": "circuits",
                "localField": "race.circuitId",
                "foreignField": "circuitId",
                "as": "circuit"
            }
        },
        {"$unwind": "$circuit"},

        # group by circuit type
        {
            "$group": {
                "_id": "$circuit.type",  # Street o Permanent
                "totalResults": {"$sum": 1},
                "dnfCount": {
                    "$sum": {
                        "$cond": [
                            {"$not": [{"$in": ["$status", ["Finished", "Lapped"]]}]}, 1, 0
                        ]
                    }
                }
            }
        },
        # calculate dnf rate
        {
            "$project": {
                "totalResults": 1,
                "dnfCount": 1,
                "dnfPercent": {"$multiply": [{"$divide": ["$dnfCount", "$totalResults"]}, 100]}
            }
        }
    ]
    return list(db.results.aggregate(pipeline))

print("\nDNF Street vs Permanent Circuits")
for c in dnf_street_vs_permanent():
    print(f"{c['_id']} - DNF: {c['dnfCount']} / {c['totalResults']} ({c['dnfPercent']:.2f}%)")


DNF Street vs Permanent Circuits
Street circuit - DNF: 26 / 180 (14.44%)
Race circuit - DNF: 46 / 419 (10.98%)
